# Spatial-cell analysis

Run with the remote `~/.virtualenvs/rfmapping` kernel on `hhw9l84`. Configure the recording and camera output below, then run the analysis to save numeric results and the single-time-bin `.rfmap`. Plot saved results with `spatial_cell_plotting.ipynb`.


In [ ]:
from pathlib import Path

import spatial_cell_analysis as analysis


## Recording and camera output

Set exactly one output bool to `True`; the camera type is never inferred. Saved camera timestamps are used when available. Otherwise ADC pulse midpoints are read as in the tuning-curve notebooks. The ADC channel is zero-based, and the threshold is in raw int16 ADC units.


In [ ]:
analysis.recording_root = Path("/mnt/senzailab/Kai/#Recording/m19")
analysis.date = "260831"
analysis.recording_number = 2
analysis.probe_name = "A"
analysis.phase_key = "baseline"

basler_output = True
optihub2_output = False
camera_input_channel = 1
camera_ttl_threshold = 14000

unit_ids = None  # None: all good units; or a list such as [7, 9].
workers = 4
result_directory = (
    analysis.recording_root / analysis.date
    / f"{analysis.date}_{analysis.recording_number}"
    / "data" / "spatial_cells" / f"Probe{analysis.probe_name}" / analysis.phase_key
)


## Arena and map parameters

The pose CSV must contain `frame`, `center_x`, `center_y`, and `hd_deg` in the existing image-coordinate convention. Raw Motive CSV exports must first be processed into these columns.

For this setup, Basler uses the midpoint of each complete **low** pulse (opto-coupled `ExposureActive`, without additional `LineInverter` inversion); OptiHub2 uses **high** pulse midpoints. Low intervals at the ADC boundaries are not Basler frames. These are explicit presets, not signal detection. See [Basler output levels](https://docs.baslerweb.com/line-status#opto-coupled-output-line) and [LineInverter](https://docs.baslerweb.com/line-inverter).

Only OptiHub2 permits the single trailing Motive frame without a TTL, as in `tuning_curves.ipynb`.


In [ ]:
analysis.x_min, analysis.x_max = 370, 920
analysis.y_min, analysis.y_max = 210, 760
analysis.rig_size_cm = 41
analysis.cm_per_px = analysis.rig_size_cm / (analysis.x_max - analysis.x_min)

analysis.theta_bin_deg = 6
analysis.number_of_distance_bins = 20
analysis.number_of_spatial_bins = 40
analysis.egocentric_smoothing_sigma = 5
analysis.allocentric_smoothing_sigma = 1.5


## Analyze and save

The output directory must be new or empty. Use another `result_directory` to keep a second run. `metadata.json` is written after all units finish; plotting requires that completed manifest.


In [ ]:
metadata = analysis.run_analysis(
    output=result_directory,
    units=unit_ids,
    workers=workers,
    basler_output=basler_output,
    optihub2_output=optihub2_output,
    camera_input_channel=camera_input_channel,
    camera_ttl_threshold=camera_ttl_threshold,
)


In [ ]:
print(f"Saved {len(metadata['unit_ids'])} units to {result_directory}")
print("session.npz, units/<id>.npz, metadata.json, egocentric_rate_map.rfmap")
metadata["source"]["camera_timing"]
